# 安裝

In [ ]:
# Qwen3.5 需要 transformers v5+
!pip install --upgrade transformers>=5.0.0
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
!pip install --no-deps unsloth

# 載入模型

Qwen3.5 0.8B — 注意事項：
- **不使用 4-bit 量化**（Unsloth 建議 Qwen3.5 使用 bfloat16 LoRA，避免量化不一致問題）
- **關閉 thinking 模式**（用於標準 SFT 微調）

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-0.8B-Instruct",
    max_seq_length = 2048,   # 上下文長度
    load_in_4bit = False,    # Qwen3.5 不建議使用 4-bit 量化
    dtype = torch.bfloat16,  # bfloat16 精度
)

# PEFT — LoRA 設定

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,

    # === LoRA 核心參數 ===
    r = 8,           # LoRA rank：控制低秩矩陣維度，常用 8-16
    lora_alpha = 8,  # 縮放因子，建議設為 r 的 1-2 倍
    lora_dropout = 0,
    bias = "none",

    # === 目標模組 ===
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],

    use_gradient_checkpointing = "unsloth",  # 節省記憶體
    random_state = 3407,
)

# 準備 Chat Template

Qwen3.5 使用 ChatML 格式，與 Qwen2.5 相同。
`enable_thinking=False` 關閉推理模式，用於標準 SFT。

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",  # Qwen3.5 使用相同的 ChatML 格式
)

# 載入資料集

In [ ]:
from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

In [ ]:
dataset[100]

# 資料標準化

In [ ]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

套用 ChatML 格式到對話資料（Qwen3.5 不需要移除 BOS token，與 Gemma-3 不同）

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    # Qwen3.5: 不需要 .removeprefix('<bos>')，ChatML 格式不使用 BOS
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,
            enable_thinking = False,  # 關閉推理模式，用於標準 SFT
        )
        for convo in convos
    ]
    return { "text": texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(dataset[100]["text"])

# 訓練模型

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,

    args = SFTConfig(
        dataset_text_field = "text",
        dataset_num_proc = 2,

        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # 有效批次 = 2 × 4 = 8

        max_steps = 30,                    # 快速測試用；正式訓練改為 num_train_epochs

        learning_rate = 2e-4,
        warmup_steps = 5,
        lr_scheduler_type = "linear",

        optim = "adamw_8bit",
        weight_decay = 0.01,

        bf16 = True,   # Qwen3.5 使用 bfloat16
        fp16 = False,

        logging_steps = 1,
        report_to = "none",
        seed = 3407,
    ),
)

只對 assistant 輸出計算 Loss，忽略 user 輸入（ChatML 格式的 marker）

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",      # Qwen ChatML user marker
    response_part = "<|im_start|>assistant\n",    # Qwen ChatML assistant marker
)

In [ ]:
# 驗證：labels 應只在 assistant 回應部分有值（非 -100）
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

In [ ]:
# 驗證 labels masking：user 輸入部分應顯示為空白
tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]
).replace(tokenizer.pad_token, " ")

# 顯示訓練前記憶體用量

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

# 開始訓練

In [ ]:
trainer_stats = trainer.train()

# 顯示訓練後記憶體用量

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## 批次輸出測試

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

messages = [{
    "role": "user",
    "content": "請接續這個序列: 1, 1, 2, 3, 5, 8,"
}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,  # 關閉推理模式
)

FastLanguageModel.for_inference(model)  # 切換到推理模式

outputs = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64,
    temperature = 0.7,
    top_p = 0.8,
    top_k = 20,
)
print(tokenizer.batch_decode(outputs)[0])

## 串流輸出測試

In [ ]:
from transformers import TextStreamer

messages = [{
    "role": "user",
    "content": "為何天空是藍色的？請用繁體中文回答。"
}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,
)

_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 128,
    temperature = 0.7,
    top_p = 0.8,
    top_k = 20,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

# 存儲模型

In [ ]:
model.save_pretrained("qwen3.5-0.8b-finetuned")
tokenizer.save_pretrained("qwen3.5-0.8b-finetuned")